# Break Through Tech AI: Nestlé 1A Group
## Stage 1: Building the DataFrame

As of 09/06/2025, the Nestlé 1A group has decided to use a subset of the [Amazon Reviews](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) dataset collected in 2023 by McAuley Lab. The entire dataset consists of 571.54 M examples. We are using the raw data from the "Grocery and Gourmet Food" category, which consists of over 14 M examples, to build an initial dataframe to then preprocess and develop machine learning models from.

Note: Outputs have been cleared due to rendering issues. Please run on your personal machine.

#### Step 0. Update, Install, and Import Python Libraries

In [2]:
#%pip install --upgrade pip
#%pip install -q datasets huggingface_hub pyarrow pandas
#%pip install matplotlib
#%pip install seaborn
#%pip install scikit-learn
#%pip install seaborn

In [1]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset

import pandas as pd
import pyarrow as pa
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

#### Step 1. Upload User Reviews file from HuggingFace

In [2]:
REV_PATH = "raw/review_categories/Grocery_and_Gourmet_Food.jsonl"

In [3]:
rev_file = hf_hub_download(repo_id="McAuley-Lab/Amazon-Reviews-2023", filename=REV_PATH, repo_type="dataset",)

In [5]:
ds_rev = load_dataset("json", data_files=rev_file, split="train")

In [ ]:
# Note: This cell may take a while to run.
df_rev = ds_rev.to_pandas()

#### Step 2. Upload Item Metadata file from HuggingFace

In [ ]:
META_PATH = "raw/meta_categories/meta_Grocery_and_Gourmet_Food.jsonl"

In [ ]:
# Filters down to only products with reviews
needed_parent_asin = set(df_rev["parent_asin"].unique())

In [ ]:
# Loading metadata
meta_file = hf_hub_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    filename=META_PATH,
    repo_type="dataset",
)


In [ ]:
#selected few columns from meta data, but you can explore more here
meta_rows = []
with open(meta_file, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Parsing meta file"):
        obj = json.loads(line)
        row = {
            "parent_asin": obj.get("parent_asin"),
            "title_meta": obj.get("title"),
            "main_category": obj.get("main_category"),
            "average_rating": obj.get("average_rating"),
            "rating_number": obj.get("rating_number"),
            "price": obj.get("price"),
            "details": obj.get("details")  # keep raw JSON dict for now
        }
        # Filters down to only products with reviews
        if row["parent_asin"] not in needed_parent_asin:
            continue
        meta_rows.append(row)

In [ ]:
df_meta = pd.DataFrame(meta_rows)

#### Step 3. Preliminary Data Analysis

In [ ]:
# Display the shape of df -- that is, the number of rows and columns.
df_rev.shape

In [ ]:
# Display the first few rows of the dataframe
df_rev.head()

In [ ]:
# Display the data types of all columns
df_rev.dtypes

#### Step 4. Data Cleaning: Remove Duplicates & Handle Missing Values


In [ ]:
print("Shape before dropping duplicates:", df_rev.shape)

In [ ]:
# Check for duplicate reviews based on user_id, asin, and text
dup_count = df_rev.duplicated(subset=["user_id", "asin", "text"]).sum()
print("Total duplicate reviews:", dup_count)

In [ ]:
# Drop duplicate reviews
df_rev = df_rev.drop_duplicates(subset=["user_id", "asin", "text"])
print("Shape after dropping duplicates:", df.shape)

In [ ]:
# Count missing ratings
print("Missing ratings:", df_rev['rating'].isna().sum())

In [17]:
# Remove missing or empty review text
df_rev = df_rev.dropna(subset=['text'])
df_rev = df_rev[df_rev['text'].str.strip() != '']

In [18]:
# Fill missing helpful_vote with 0
df_rev['helpful_vote'] = df_rev['helpful_vote'].fillna(0)

# Drop rows with missing verified_purchase
df_rev = df_rev.dropna(subset=['verified_purchase'])

In [ ]:
print("Final dataset shape:", df_rev.shape)
print(df_rev.isna().sum())

#### Data Fields for User Reviews

| Field            | Type   | Explanation |
| :--------------- | :----- | :---------- |
| rating           | float  | Rating of the product (from 1.0 to 5.0). |
| text             | str    | Text body of the user review. |
| images           | list   | Images that users post after they have received the product.<br><br>Note: Each image has different sizes (small, medium, large), represented by the `small_image_url`, `medium_image_url`, and `large_image_url` respectively. |
| asin             | str    | ID of the product. |
| parent_asin      | str    | Parent ID of the product.<br><br>Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. Please use parent ID to find product meta. |
| user_id          | str    | ID of the reviewer. |
| timestamp        | int    | Time of the review (unix time). |
| verified_purchase| bool   | User purchase verification. |
| helpful_vote     | int    | Helpful votes of the review. |